In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [2]:
YEAR = 2023
API_KEY = "cf4fd28398432786abcfab6be3868157c4836d1d"

# NAME = "denver"
# STATE = "08"
# COUNTIES = [
#     "031",  # Denver
#     "005",  # Arapahoe
#     "059",  # Jefferson
#     # "013",  # Boulder
#     "035",  # Douglas
#     "001",  # Adams
# ]
# COUNTY_NAMES = [
#     "City and County of Denver, Colorado, USA",
#     "Arapahoe County, Colorado, USA",
#     "Jefferson County, Colorado, USA",
#     # "Boulder County, Colorado, USA",
#     "Douglas County, Colorado, USA",
#     "Adams County, Colorado, USA",
# ]

# NAME = "new_york"
# STATE = "36"
# COUNTIES = [
#     "005",  # Bronx
#     "047",  # Brooklyn (Kings)
#     "061",  # Manhattan
#     "081",  # Queens
#     # "085"   # Staten Island
#         ]
# COUNTY_NAMES = [
#     "Bronx County, New York, USA",
#     "Kings County, New York, USA",
#     "New York County, New York, USA",
#     "Queens County, New York, USA",
#     # "Richmond County, New York, USA",
# ]


# NAME = "atlanta"
# STATE = "13"
# COUNTIES = [
#             "121",  # Fulton
#             "089",  # DeKalb
#             "067",  # Cobb
#             "135",  # Gwinnett
# ]
# COUNTY_NAMES = [
#         "Fulton County, Georgia, USA",
#         "DeKalb County, Georgia, USA",
#         "Cobb County, Georgia, USA",
#         "Gwinnett County, Georgia, USA",
#     ]

NAME = "boston"
STATE = "25"
COUNTIES =  [
    "025",  # Suffolk
    "009",  # Essex
    "017",  # Middlesex
    "021",  # Norfolk
    # "023"   # Plymouth
]
COUNTY_NAMES = [
    "Suffolk County, Massachusetts, USA",
    "Middlesex County, Massachusetts, USA",
    "Norfolk County, Massachusetts, USA",
    "Essex County, Massachusetts, USA",
    # "Plymouth County, Massachusetts, USA",
]

In [3]:
from src.acs import (
    load_acs_data,
    compute_acs_features
)

acs = load_acs_data(
    state_fips=STATE,
    county_fips_list=COUNTIES,
    year=YEAR,
    census_api_key=API_KEY
)

acs = compute_acs_features(acs)

acs = acs.drop(columns=["index"])
acs["state_fips"] = acs["tract_id"].str[:2]
acs["county_fips"] = acs["tract_id"].str[2:5]
acs["tract_code"] = acs["tract_id"].str[5:]

In [4]:
from src.geometry import (
    load_tract_geometries,
    add_cbd_distance
)
from src.road_network import compute_network_features

In [5]:
geometry = load_tract_geometries(
    state_fips=STATE,
    county_fips_list=COUNTIES
)


Loading tract geometries for state 25...
Raw tracts: 1620
Filtered tracts: 920
Geometry table ready.


In [6]:
geometry = compute_network_features(
    geometry,
    county_names=COUNTY_NAMES
)

In [7]:
print(geometry.columns)

Index(['tract_id', 'state_fips', 'county_fips', 'tract_code', 'land_area_km2',
       'centroid_lat', 'centroid_lon', 'geometry', 'node_density',
       'intersection_density', 'street_density'],
      dtype='object')


In [8]:
from src.amenities import compute_amenity_features

geometry = compute_amenity_features(
    geometry,
    counties=COUNTY_NAMES
)

Processing: restaurant
Processing: cafe
Processing: school
Processing: hospital
Processing: pharmacy
Processing: supermarket
Processing: park
Processing: transit_stop


In [9]:
from src.geometry import (
    load_tract_geometries,
    add_cbd_distance
)

geometry = add_cbd_distance(geometry, city_name=NAME)

print(geometry.columns.tolist())

['tract_id', 'state_fips', 'county_fips', 'tract_code', 'land_area_km2', 'centroid_lat', 'centroid_lon', 'geometry', 'node_density', 'intersection_density', 'street_density', 'restaurant_count', 'restaurant_density', 'cafe_count', 'cafe_density', 'school_count', 'school_density', 'hospital_count', 'hospital_density', 'pharmacy_count', 'pharmacy_density', 'supermarket_count', 'supermarket_density', 'park_count', 'park_density', 'transit_stop_count', 'transit_stop_density', 'distance_to_cbd_km']


In [10]:
tracts = geometry.merge(
    acs,
    on="tract_id",
    how="left"
)

print(len(tracts))

print(
    tracts["tract_id"]
    .nunique()
)

print(
    tracts["total_population"]
    .isna()
    .sum()
)

920
920
0


In [11]:
print(geometry.columns.tolist())

['tract_id', 'state_fips', 'county_fips', 'tract_code', 'land_area_km2', 'centroid_lat', 'centroid_lon', 'geometry', 'node_density', 'intersection_density', 'street_density', 'restaurant_count', 'restaurant_density', 'cafe_count', 'cafe_density', 'school_count', 'school_density', 'hospital_count', 'hospital_density', 'pharmacy_count', 'pharmacy_density', 'supermarket_count', 'supermarket_density', 'park_count', 'park_density', 'transit_stop_count', 'transit_stop_density', 'distance_to_cbd_km']


In [12]:
print(tracts.columns.tolist())

['tract_id', 'state_fips_x', 'county_fips_x', 'tract_code_x', 'land_area_km2', 'centroid_lat', 'centroid_lon', 'geometry', 'node_density', 'intersection_density', 'street_density', 'restaurant_count', 'restaurant_density', 'cafe_count', 'cafe_density', 'school_count', 'school_density', 'hospital_count', 'hospital_density', 'pharmacy_count', 'pharmacy_density', 'supermarket_count', 'supermarket_density', 'park_count', 'park_density', 'transit_stop_count', 'transit_stop_density', 'distance_to_cbd_km', 'total_population', 'median_age', 'male_population', 'female_population', 'households', 'avg_household_size', 'm_under_5', 'm_5_9', 'm_10_14', 'm_15_17', 'f_under_5', 'f_5_9', 'f_10_14', 'f_15_17', 'm_65_66', 'm_67_69', 'm_70_74', 'm_75_79', 'm_80_84', 'm_85_plus', 'f_65_66', 'f_67_69', 'f_70_74', 'f_75_79', 'f_80_84', 'f_85_plus', 'median_income', 'per_capita_income', 'poverty_total', 'below_poverty', 'labor_force', 'employed', 'unemployed', 'snap_households', 'public_assistance_households

In [13]:
from src.model_dataset import (
    build_modeling_dataset
)

model_df = build_modeling_dataset(
    tracts
)

print(model_df.columns.tolist())

['tract_id', 'total_population', 'population_density', 'pct_under_18', 'pct_18_64', 'pct_over_65', 'sex_ratio', 'median_age', 'avg_household_size', 'households', 'median_household_income', 'per_capita_income', 'poverty_rate', 'unemployment_rate', 'labor_force_participation_rate', 'snap_participation_rate', 'public_assistance_rate', 'pct_high_school', 'pct_bachelors_degree', 'pct_graduate_degree', 'pct_no_vehicle_households', 'pct_one_vehicle_households', 'pct_two_plus_vehicle_households', 'median_commute_time', 'pct_public_transit_commute', 'pct_car_commute', 'pct_walk_commute', 'pct_bike_commute', 'pct_work_from_home', 'housing_units', 'occupied_housing_units', 'vacant_housing_units', 'homeownership_rate', 'median_rent', 'median_home_value', 'housing_density', 'median_year_built', 'land_area_km2', 'distance_to_cbd_km', 'pct_white_non_hispanic', 'pct_black', 'pct_hispanic_latino', 'pct_asian', 'pct_other_multiracial', 'intersection_density', 'node_density', 'street_density', 'restauran

/home/beau/Documents/thesis/src/model_dataset.py:12: RuntimeWarning: invalid value encountered in divide
  a / b


In [14]:
print(model_df.head())

      tract_id  total_population  population_density  pct_under_18  pct_18_64  \
0  25017336202              3223         1658.323608      0.220602   0.523736   
1  25017330102              5118          281.530212      0.261626   0.583431   
2  25017339101              3091         1889.363770      0.095762   0.599159   
3  25017356703              3154         1137.221069      0.176601   0.544705   
4  25017330202              3970          278.604401      0.189421   0.657431   

   pct_over_65  sex_ratio  median_age  avg_household_size  households  ...  \
0     0.255662   0.829171   46.900002                2.39        1298  ...   
1     0.154943   0.943052   44.900002                2.78        1831  ...   
2     0.305079   0.743373   47.900002                1.73        1785  ...   
3     0.278694   0.815774   51.500000                2.11        1475  ...   
4     0.153149   1.189741   40.400002                2.39        1656  ...   

   hospital_count  hospital_density  pharmac

In [15]:
from pathlib import Path

output_dir = Path("../data/processed_predictors")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

model_df.to_parquet(
    output_dir / f"{NAME}_{YEAR}_predictor_dataset.parquet",
    index=False
)